In [ ]:
import os
import glob
import re
import pandas as pd
import numpy as np
import aqi
from sqlalchemy import create_engine
from sqlalchemy.dialects.postgresql import insert
from dotenv import load_dotenv

In [ ]:
DATA_DIR = "new_data/air"
load_dotenv() 
DB_URL = os.getenv("DB_URL")

NAME_NORMALIZATION = {
    "Bolbes": "Volvi",
    "Ampelokepon-Menemenes": "Ampelokipoi-Menemeni",
    "Ampelokipon": "Ampelokipoi-Menemeni",
    "Thermaikou": "Thermaikos",
    "Khalkedonos": "Chalkidona",
    "Chalkidonos": "Chalkidona",
    "Kordeliou": "Kordelio-Evosmos",
    "Kordeliou-Euosmou": "Kordelio-Evosmos",
    "Lagkada": "Lagkadas",
    "Neapoles-Sykeon": "Neapoli-Sykies",
    "Neapoli": "Neapoli-Sykies",
    "Pavlou_Mela": "Pavlos Melas",
    "Thermes": "Thermi",
    "Thessalonikes": "Thessaloniki"

}

In [ ]:
def calculate_hourly_aqi(row):
    try:
        sub_indices = []
        
        #Helper function to check both old ('o3') and new ('o3_conc') column names
        def get_val(base_name):
            val = row.get(f"{base_name}_conc")
            if pd.isna(val):
                val = row.get(base_name)
            return float(val) if pd.notna(val) else None

        pm25 = get_val('pm25')
        if pm25 is not None:
            sub_indices.append(aqi.to_iaqi(aqi.POLLUTANT_PM25, str(pm25)))
            
        pm10 = get_val('pm10')
        if pm10 is not None:
            sub_indices.append(aqi.to_iaqi(aqi.POLLUTANT_PM10, str(pm10)))
            
        no2 = get_val('no2')
        if no2 is not None:
            no2_ppb = (no2 * 24.45) / 46.01
            sub_indices.append(aqi.to_iaqi(aqi.POLLUTANT_NO2_1H, str(no2_ppb))) 
            
        co = get_val('co')
        if co is not None:
            co_ppm = (co * 24.45) / (28.01 * 1000)
            sub_indices.append(aqi.to_iaqi(aqi.POLLUTANT_CO_8H, str(co_ppm)))
            
        o3 = get_val('o3')
        if o3 is not None:
            o3_ppm = (o3 * 24.45) / (48.00 * 1000)
            sub_indices.append(aqi.to_iaqi(aqi.POLLUTANT_O3_8H, str(o3_ppm)))
            
        so2 = get_val('so2')
        if so2 is not None:
            so2_ppb = (so2 * 24.45) / 64.06
            sub_indices.append(aqi.to_iaqi(aqi.POLLUTANT_SO2_1H, str(so2_ppb)))
            
        return max(sub_indices) if sub_indices else np.nan
        
    except Exception as e:
        return np.nan

In [ ]:
file_pattern = os.path.join(DATA_DIR, "*", "municipality_of_*_pollutants_conc_timeseries-yearly_*.csv")
all_files = glob.glob(file_pattern)

if not all_files:
    print(f"No CSV files found matching the pattern in '{DATA_DIR}' subfolders!")

all_monthly_summaries = []

#Iterates over the explicitly matched files instead of doing another recursive search
for file_path in all_files:
    filename = os.path.basename(file_path)
    
    match = re.search(r"municipality_of_(.+)_pollutants_conc_timeseries-yearly", filename)
    if not match:
        continue
        
    raw_municipality = match.group(1).title()
    municipality = NAME_NORMALIZATION.get(raw_municipality, raw_municipality)
    
    df = pd.read_csv(file_path)
    
    #Normalizes column names for newer datasets that use the "_conc" suffix
    df.rename(columns={
        'co_conc': 'co',
        'no2_conc': 'no2',
        'so2_conc': 'so2',
        'o3_conc': 'o3'
    }, inplace=True)
    
    df['date'] = pd.to_datetime(df['time'])
    
    #Calculates AQI
    df['hourly_aqi'] = df.apply(calculate_hourly_aqi, axis=1)
    
    monthly_summary = df.groupby([df['date'].dt.year.rename('year'), 
                                  df['date'].dt.month.rename('month')])['hourly_aqi'].mean().reset_index()
    monthly_summary['mean_aqi'] = monthly_summary['hourly_aqi'].round(0)
    monthly_summary['municipality'] = municipality
    
    final_df = monthly_summary[['municipality', 'year', 'month', 'mean_aqi']]
    final_df = final_df.dropna(subset=['mean_aqi'])
    
    all_monthly_summaries.append(final_df)


In [ ]:
if all_monthly_summaries:
    master_df = pd.concat(all_monthly_summaries, ignore_index=True)



In [ ]:
#Custom insertion method to ignore duplicates
def insert_do_nothing(table, conn, keys, data_iter):
    #Zips the column names and the data rows into a list of dictionaries
    data = [dict(zip(keys, row)) for row in data_iter]
    
    #Creates the standard insert statement
    insert_stmt = insert(table.table).values(data)

    do_nothing_stmt = insert_stmt.on_conflict_do_nothing(
        index_elements=['municipality', 'year', 'month']
    )
    
    result = conn.execute(do_nothing_stmt)
    return result.rowcount

#Connects and pushes to db
engine = create_engine(DB_URL)

master_df.to_sql(
    'historical_aqi', 
    engine, 
    if_exists='append', 
    index=False, 
    method=insert_do_nothing
)